# Building a Scientific Agent: Particle Size Analysis

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

In this notebook you'll build an **AI agent from scratch** that takes a
microscopy image and answers questions about the particles in it: how many,
how big, how broad the size distribution. This is the "hello world" of
quantitative microscopy — universal across SEM, TEM, optical, and AFM —
which makes it a great canvas for understanding what an agent actually *is*.

## What an agent is (and isn't)

> **Agent** = LLM + tools + a loop that lets the LLM call the tools.

That's it. There is no neural-network magic happening at the agent layer.
The model is the *planner*; your Python functions are the *hands*. The loop
just shuttles messages back and forth until the model has nothing left to do.

## What you'll build

1. A synthetic microscopy image you can see and verify.
2. Four tools wrapping standard image-processing operations:
   `inspect_image`, `threshold_image`, `segment_particles`, `measure_particles`.
3. A from-scratch agent loop that hands those tools to Claude.
4. The agent will answer prompts like *"How many particles are in this image,
   and what's the mean diameter?"* — by autonomously deciding which tools to
   call, in which order.

No frameworks. ~150 lines of agent code. Once you've seen this loop, every
fancier framework (LangGraph, AutoGen, CrewAI…) is just nicer packaging.

---
## Setup

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install anthropic numpy scipy scikit-image matplotlib

In [ ]:
import os, getpass, json, io, base64
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6" 

---
## Part 1: Make a synthetic microscopy image

So everyone runs the same data and we know the *ground truth* size
distribution. Real workflows would load a SEM/TEM/optical micrograph instead
— the agent code is identical.

In [ ]:
def synthesize_image(n_particles=45, img_size=512, mean_radius=14,
                     radius_std=4, noise_sigma=0.04, seed=0,
                     min_gap=4):
    """Make a fake microscopy image: bright, non-overlapping circular particles."""
    rng = np.random.default_rng(seed)
    img = np.zeros((img_size, img_size), dtype=np.float32)
    yy, xx = np.mgrid[:img_size, :img_size]
    placed = []                                          # list of (cx, cy, r)
    attempts, max_attempts = 0, n_particles * 200
    while len(placed) < n_particles and attempts < max_attempts:
        attempts += 1
        r = float(np.clip(rng.normal(mean_radius, radius_std), 6, 28))
        cx, cy = rng.uniform(r+2, img_size-r-2, size=2)
        # Reject if this circle would touch an existing one
        if any((cx-px)**2 + (cy-py)**2 < (r + pr + min_gap)**2 for px, py, pr in placed):
            continue
        placed.append((cx, cy, r))
    for cx, cy, r in placed:
        mask = (xx - cx)**2 + (yy - cy)**2 <= r**2
        img[mask] = np.maximum(img[mask], 0.85 + 0.05*rng.standard_normal())
    img += noise_sigma * rng.standard_normal(img.shape)
    img = np.clip(img, 0, 1)
    true_radii = np.array([r for _, _, r in placed])
    return img, true_radii

IMAGE, TRUE_RADII = synthesize_image()
print(f"Image: {IMAGE.shape}, intensity range [{IMAGE.min():.2f}, {IMAGE.max():.2f}]")
print(f"Ground truth: {len(TRUE_RADII)} particles, "
      f"mean radius {TRUE_RADII.mean():.1f} px (std {TRUE_RADII.std():.1f})")

plt.figure(figsize=(5,5))
plt.imshow(IMAGE, cmap="gray"); plt.title("Synthetic microscopy image"); plt.axis("off")
plt.show()

---
## Part 2: Design the agent's tools

An agent is only as capable as its tools. We design four — each does one
thing, with explicit JSON-Schema inputs/outputs so Claude can reason about
them without seeing the implementation.

The handler dictionary at the bottom is what the agent actually invokes
when the LLM emits a tool call.

In [ ]:
# A small in-process "session" so tools can pass results to each other.
SESSION = {"image": IMAGE, "binary": None, "labels": None, "n_labels": 0}

# ---- Tool 1: inspect_image -----------------------------------------------
def inspect_image() -> dict:
    img = SESSION["image"]
    return {
        "shape":         list(img.shape),
        "dtype":         str(img.dtype),
        "intensity_min": float(img.min()),
        "intensity_max": float(img.max()),
        "intensity_mean": float(img.mean()),
        "intensity_std":  float(img.std()),
        "histogram_bin_edges": np.linspace(0, 1, 11).round(2).tolist(),
        "histogram_counts":    np.histogram(img, bins=np.linspace(0, 1, 11))[0].tolist(),
    }

# ---- Tool 2: threshold_image ---------------------------------------------
def threshold_image(method: str = "otsu", value: float | None = None) -> dict:
    """Convert the grayscale image into a binary (foreground / background) mask."""
    img = SESSION["image"]
    if method == "otsu":
        # Compact Otsu's method (no scikit-image needed)
        hist, edges = np.histogram(img, bins=256, range=(0, 1))
        p = hist / hist.sum()
        omega = np.cumsum(p)
        mu = np.cumsum(p * (edges[:-1] + 0.5*(edges[1]-edges[0])))
        mu_t = mu[-1]
        sigma_b = (mu_t * omega - mu)**2 / (omega * (1 - omega) + 1e-12)
        thr = float(edges[:-1][np.argmax(sigma_b)])
    elif method == "fixed":
        if value is None:
            return {"error": "method='fixed' requires a value (0-1)"}
        thr = float(value)
    else:
        return {"error": f"unknown method {method!r}; use 'otsu' or 'fixed'"}
    SESSION["binary"] = (img > thr).astype(np.uint8)
    return {"method": method, "threshold": thr,
            "foreground_fraction": float(SESSION["binary"].mean())}

# ---- Tool 3: segment_particles -------------------------------------------
def segment_particles(min_area: int = 20) -> dict:
    """Connected-component labeling on the binary mask."""
    if SESSION["binary"] is None:
        return {"error": "Call threshold_image first."}
    labels, n = ndimage.label(SESSION["binary"])
    if min_area > 0:
        sizes = ndimage.sum(SESSION["binary"], labels, range(1, n+1))
        keep = np.where(sizes >= min_area)[0] + 1
        # Re-label so kept particles are 1..k
        relabel = np.zeros(n+1, dtype=int)
        relabel[keep] = np.arange(1, len(keep)+1)
        labels = relabel[labels]
        n = len(keep)
    SESSION["labels"], SESSION["n_labels"] = labels, n
    return {"n_particles": int(n), "min_area_used": int(min_area)}

# ---- Tool 4: measure_particles -------------------------------------------
def measure_particles() -> dict:
    """Compute area and equivalent radius for each labelled particle."""
    if SESSION["labels"] is None or SESSION["n_labels"] == 0:
        return {"error": "Call segment_particles first."}
    labels, n = SESSION["labels"], SESSION["n_labels"]
    areas = ndimage.sum(np.ones_like(labels), labels, range(1, n+1))
    radii_eq = np.sqrt(areas / np.pi)             # equivalent radius (px)
    return {
        "n_particles":      int(n),
        "mean_radius_px":   float(radii_eq.mean()),
        "median_radius_px": float(np.median(radii_eq)),
        "std_radius_px":    float(radii_eq.std()),
        "min_radius_px":    float(radii_eq.min()),
        "max_radius_px":    float(radii_eq.max()),
        "histogram_bin_edges_px": np.linspace(0, radii_eq.max()+2, 11).round(1).tolist(),
        "histogram_counts":       np.histogram(radii_eq,
                                               bins=np.linspace(0, radii_eq.max()+2, 11))[0].tolist(),
    }

# Quick smoke test
print("inspect_image ->", inspect_image())
print("threshold_image('otsu') ->", threshold_image("otsu"))
print("segment_particles(min_area=30) ->", segment_particles(min_area=30))
print("measure_particles() ->", measure_particles())

**Note:** Tools share state through `SESSION`. That's the simplest
pattern; in a production system you'd pass an object explicitly or use a
proper data store. The principle is the same: tool calls form a *pipeline*
where later steps consume earlier results.

---
## Part 3: Tool schemas (what the LLM sees)

The LLM never sees the Python code above. It sees these JSON schemas.
Naming and descriptions matter a lot — they're the only signal Claude has
about *when* to call each tool.

In [ ]:
TOOLS = [
    {
        "name": "inspect_image",
        "description": "Return basic statistics and an intensity histogram of the loaded image. "
                       "Use this first to get a feel for the data.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "threshold_image",
        "description": "Binarize the image (foreground vs background). Choose 'otsu' for "
                       "automatic thresholding, or 'fixed' with a value in [0,1].",
        "input_schema": {
            "type": "object",
            "properties": {
                "method": {"type": "string", "enum": ["otsu", "fixed"], "default": "otsu"},
                "value":  {"type": ["number", "null"],
                           "description": "Threshold value (0-1) when method='fixed'."},
            },
            "required": ["method"],
        },
    },
    {
        "name": "segment_particles",
        "description": "Run connected-component labelling on the binarized image to count "
                       "individual particles. Filters out noise blobs smaller than min_area.",
        "input_schema": {
            "type": "object",
            "properties": {"min_area": {"type": "integer", "default": 20,
                                        "description": "Minimum particle area in pixels."}},
            "required": [],
        },
    },
    {
        "name": "measure_particles",
        "description": "Compute area, equivalent radius, and a size histogram for each segmented "
                       "particle. Call after segment_particles.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
]

# Map each tool name to its Python implementation.
HANDLERS = {
    "inspect_image":     inspect_image,
    "threshold_image":   threshold_image,
    "segment_particles": segment_particles,
    "measure_particles": measure_particles,
}

---
## Part 4: The agent loop

The whole agent is this one function. Read it carefully — every "agent
framework" you'll ever use is a more elaborate version of these 25 lines.

In [ ]:
def run_agent(user_question: str, system_prompt: str = "", max_turns: int = 8,
              verbose: bool = True) -> str:
    messages = [{"role": "user", "content": user_question}]
    for turn in range(max_turns):
        resp = client.messages.create(
            model=MODEL, max_tokens=1024,
            system=system_prompt, tools=TOOLS, messages=messages,
        )
        # Always append the full assistant turn (text + any tool_use blocks)
        messages.append({"role": "assistant", "content": resp.content})

        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            # No more tool calls -> we have the final answer
            return "".join(b.text for b in resp.content if b.type == "text")

        # Run each tool the model asked for, feed results back as tool_results
        tool_results = []
        for tu in tool_uses:
            if verbose:
                print(f"  [{turn}] {tu.name}({json.dumps(tu.input)})")
            try:
                out = HANDLERS[tu.name](**tu.input)
            except Exception as e:
                out = {"error": str(e)}
            tool_results.append({"type": "tool_result", "tool_use_id": tu.id,
                                 "content": json.dumps(out)})
        messages.append({"role": "user", "content": tool_results})
    return "[max turns reached]"

---
## Part 5: Run the agent

Now we just ask. Notice that the user prompt doesn't tell the model *which*
tools to call or in what order — the model figures that out from the tool
descriptions.

In [ ]:
SYSTEM = ("You are a microscopy data analyst. You have tools to inspect, threshold, "
          "segment, and measure particles in an already-loaded image. Always call the "
          "tools rather than guessing. End with a concise quantitative summary.")

answer = run_agent(
    "I have a microscopy image already loaded. How many particles are in it, "
    "and what is the mean equivalent radius (in pixels)? Please also comment "
    "on the spread of the size distribution.",
    system_prompt=SYSTEM,
)
print("\n=== Agent's final answer ===\n")
print(answer)

**Trace what just happened:**

The model planned a sensible pipeline on its own:
`inspect_image` → `threshold_image('otsu')` → `segment_particles` → `measure_particles`
→ summarise.

That ordering came from *the descriptions*, not from any hard-coded plan.
Change the descriptions, and the agent's behaviour shifts. This is the
single most important practical lesson about agents — **the tool catalog is
the program**.

---
## Part 6: Visualise what the agent did

For a tutorial demo it's compelling to *see* the segmentation overlay so
participants can verify the agent's numbers visually.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(SESSION["image"], cmap="gray"); axes[0].set_title("Original")
axes[1].imshow(SESSION["binary"], cmap="gray"); axes[1].set_title("Otsu threshold")
axes[2].imshow(SESSION["labels"], cmap="nipy_spectral"); axes[2].set_title(
    f"{SESSION['n_labels']} particles labelled")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

# How well did it match ground truth?
labels, n = SESSION["labels"], SESSION["n_labels"]
areas = ndimage.sum(np.ones_like(labels), labels, range(1, n+1))
radii_eq = np.sqrt(areas / np.pi)
print(f"Ground truth: n={len(TRUE_RADII)}, mean radius={TRUE_RADII.mean():.1f} px")
print(f"Agent found : n={n}, mean radius={radii_eq.mean():.1f} px")

---
## Part 7: A harder question (forces multi-step reasoning)

Now ask something the agent has to *reason* about, not just measure. It will
need to choose parameters and re-run a tool.

In [ ]:
# Reset session for a fresh run
SESSION.update({"binary": None, "labels": None, "n_labels": 0})

answer = run_agent(
    "I'm worried that small noise blobs might be inflating my particle count. "
    "Try segmentation with two different min_area thresholds (e.g. 5 and 100), "
    "compare the resulting counts, and recommend a sensible value. Then give me "
    "the final particle count and size statistics at your recommended setting.",
    system_prompt=SYSTEM,
)
print("\n=== Agent's final answer ===\n")
print(answer)

Here you should see the agent:
- Run `segment_particles(min_area=5)` and `segment_particles(min_area=100)` separately,
- Compare the two counts in plain language,
- Pick a value and call `measure_particles` once more.

That "compare two configurations and pick one" behaviour is *not* something
we programmed. It's emergent from the model + tool catalog + a loop that
lets it call tools more than once.

---
## Takeaways

1. An agent is **a small loop**. Everything else is glue.
2. Tools are the agent's hands — **descriptions are the spec the model reads**.
3. Shared state between tools (the `SESSION` dict here) is fine for prototypes;
   for production you want explicit data passing or a proper store.
4. Always *visualise* the intermediate state so you can verify the agent's
   numerical claims — agents fail silently more often than loudly.

## Where to go next

- **Notebook 05 (MCP)** — wrap these same tools in an MCP server so any
  agent (Claude Desktop, Cursor, ChatGPT, …) can call them.
- **Notebook 06 (Multi-agent + Eval + Caching)** — when one agent isn't
  enough, and how to actually measure whether your agent works.
- **Notebook 07 (LangGraph)** — the same loop you just wrote, organised as
  an explicit state graph with persistence and conditional branches.